# Corridor Transit Profiles — Calibrated-Survey ("hybrid") vs Ticketing-Based

Compares the two transit demand profiles along the LRT corridor, link by link and direction by direction:

| Profile | Bus layer | Rail layer | Frame |
|---|---|---|---|
| **Hybrid (calibrated survey)** — `Corridor_flow_profile_survey_2022.ipynb` | survey Public Bus + Matronit, destination pattern blended with RavKav × OnBoard, volumes from RavKav where ticketing coverage is credible, grown to 2022 where not; plus the survey's taxi-type modes | survey rail, door-to-door, × 0.793 | residents, doorstep origins; 2022 |
| **Ticketing-based** — `Transit_complete_matrix.ipynb` / `Vintage_alignment_2022.ipynb` | RavKav journey volumes × OnBoard destinations, May 2022 | 2019 smartcard station-to-station × 0.793 | all riders, boarding-stop origins; 2022 |

Both use the same 18-area line sequence and the same link-assignment rule (each OD pair with both ends on the line is added to every link between them). The comparison:

1. link flows of the two profiles and their components (bus, taxi-type, rail / train), per direction;
2. the intermediate steps of the calibration — raw survey bus (2018), the all-RavKav variant — to show what each step does to the profile;
3. which area pairs drive the difference on the busiest links;
4. transit share along the line for both matrix sets (transit ÷ total profile).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/ths2017/three_mode_2022'
sub_key = pd.read_excel('Input/Submatrix_tazs.xlsx'); area_of = sub_key.set_index('TAZ')['AggAreaCode']; AREA = sorted(area_of.unique())
legend = pd.read_csv('Output/ths2017/study_taz/submatrices/area_legend.csv').set_index('AggAreaCode'); names = legend['AggAreaName']
ref = pd.read_csv('Output/transit/corridor_link_flows_transit_2022.csv'); SEQ = list(ref['from_code']) + [int(ref['to_code'].iloc[-1])]

def load(path):
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
def taz_to_area(m):
    m = m.copy(); m.index.name, m.columns.name = 'o', 'd'
    long = m.stack().reset_index(); long.columns = ['o', 'd', 'v']
    long['O'] = long['o'].map(area_of); long['D'] = long['d'].map(area_of)
    return long.dropna(subset=['O', 'D']).groupby(['O', 'D'])['v'].sum().unstack().reindex(index=AREA, columns=AREA, fill_value=0).fillna(0)
A = lambda m: m.reindex(index=AREA, columns=AREA).fillna(0)

# hybrid (calibrated survey) components, 2022
H_bus = load(f'{OUT}/bus_2022_excl_taxi_taz.csv'); H_bus = taz_to_area(H_bus)
H_taxi = taz_to_area(load(f'{OUT}/taxi_2022_taz.csv'))
H_rail = A(load(f'{OUT}/rail_2022_area.csv'))
H_total = A(load(f'{OUT}/car_2022_area.csv')) + A(load(f'{OUT}/bus_2022_area.csv')) + H_rail
# ticketing-based components, 2022
T_bus = A(load('Output/bus/bus_od_area_new.csv'))
T_train = A(load('Output/train/train_od_area.csv')) * (54.7 / 69.0)
T_total = A(load('Output/transit/all_adjusted_area_2022.csv'))
# calibration steps (2018 survey bus, all-RavKav variant), for the "what each step does" view
S_bus18 = A(load('Output/ths2017/two_mode/bus_survey_sz.csv').iloc[0:0]) if False else taz_to_area(load('Output/ths2017/two_mode/bus_survey_taz.csv'))
R_bus = taz_to_area(load('Output/ths2017/two_mode/bus_calibrated_all_ravkav_taz.csv'))
print(f"sub-area totals — hybrid: bus {H_bus.values.sum():,.0f}, taxi-type {H_taxi.values.sum():,.0f}, rail {H_rail.values.sum():,.0f} | "
      f"ticketing: bus {T_bus.values.sum():,.0f}, train {T_train.values.sum():,.0f} | survey bus 2018 {S_bus18.values.sum():,.0f}, all-RavKav variant {R_bus.values.sum():,.0f}")

In [ ]:
def link_flows(m, seq=SEQ):
    sub = m.reindex(index=seq, columns=seq).fillna(0).values; n = len(seq)
    f, b = np.zeros(n - 1), np.zeros(n - 1)
    for i in range(n):
        for j in range(n):
            if i < j: f[i:j] += sub[i, j]
            elif i > j: b[j:i] += sub[i, j]
    return f, b
COMP = {'hybrid: bus (calibrated survey)': H_bus, 'hybrid: taxi-type': H_taxi, 'hybrid: rail (survey)': H_rail,
        'ticketing: bus (RavKav × OnBoard)': T_bus, 'ticketing: train (station)': T_train,
        'step: survey bus 2018 (raw)': S_bus18, 'step: bus, all-RavKav volumes': R_bus, 'total: hybrid set': H_total, 'total: ticketing set': T_total}
LF = {k: link_flows(v) for k, v in COMP.items()}
links = [f'{names[a]} – {names[b]}' for a, b in zip(SEQ[:-1], SEQ[1:])]
tbl = pd.DataFrame({'link': links})
for d, di in [('1→23', 0), ('23→1', 1)]:
    tbl[f'hybrid transit {d}'] = LF['hybrid: bus (calibrated survey)'][di] + LF['hybrid: taxi-type'][di] + LF['hybrid: rail (survey)'][di]
    tbl[f'ticketing transit {d}'] = LF['ticketing: bus (RavKav × OnBoard)'][di] + LF['ticketing: train (station)'][di]
    tbl[f'diff {d}'] = tbl[f'hybrid transit {d}'] - tbl[f'ticketing transit {d}']
    tbl[f'ratio {d}'] = tbl[f'hybrid transit {d}'] / tbl[f'ticketing transit {d}'].replace(0, np.nan)
    tbl[f'transit share hybrid {d}'] = tbl[f'hybrid transit {d}'] / LF['total: hybrid set'][di]
    tbl[f'transit share ticketing {d}'] = tbl[f'ticketing transit {d}'] / LF['total: ticketing set'][di]
assert np.allclose(tbl['ticketing transit 1→23'].round(0), ref['flow_dir_1_to_23']) and np.allclose(tbl['ticketing transit 23→1'].round(0), ref['flow_dir_23_to_1']), "ticketing profile must reproduce the saved one"
tbl.to_csv(f'{OUT}/corridor_profile_hybrid_vs_ticketing.csv', index=False, float_format='%.3f')
comp = pd.DataFrame({'link': links, **{f'{k} {d}': LF[k][di] for k in COMP for d, di in [('1→23', 0), ('23→1', 1)]}})
comp.to_csv(f'{OUT}/corridor_profile_components.csv', index=False, float_format='%.1f')
print("link flows, transit, both profiles (trips 6:00–9:00):")
tbl[['link', 'hybrid transit 1→23', 'ticketing transit 1→23', 'ratio 1→23', 'hybrid transit 23→1', 'ticketing transit 23→1', 'ratio 23→1']].round(2)

In [ ]:
sums = {}
for d, di in [('1→23', 0), ('23→1', 1)]:
    sums[d] = {'hybrid transit link-trips': tbl[f'hybrid transit {d}'].sum(), 'ticketing transit link-trips': tbl[f'ticketing transit {d}'].sum(),
               'hybrid peak': tbl[f'hybrid transit {d}'].max(), 'ticketing peak': tbl[f'ticketing transit {d}'].max(),
               'hybrid peak link': links[tbl[f'hybrid transit {d}'].idxmax()], 'ticketing peak link': links[tbl[f'ticketing transit {d}'].idxmax()],
               'Haifa segment (Tirat Carmel–Hadar) hybrid/ticketing': tbl[f'hybrid transit {d}'][:9].sum() / tbl[f'ticketing transit {d}'][:9].sum(),
               'Krayot–Nazareth segment (Namal–Nazareth) hybrid/ticketing': tbl[f'hybrid transit {d}'][10:].sum() / tbl[f'ticketing transit {d}'][10:].sum()}
pd.DataFrame(sums).T

## What each calibration step does to the profile

Raw survey bus (2018) → volumes from RavKav everywhere (all-RavKav variant) → coverage-guarded volumes + 2022 growth (the hybrid bus), against the ticketing bus. Taxi-type and rail / train are shown separately.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 11), facecolor='white', sharex=True)
x = np.arange(len(SEQ))
for ax, (d, di) in zip(axes, [('1→23', 0), ('23→1', 1)]):
    for k, color, ls, lw in [('step: survey bus 2018 (raw)', MUTED, ':', 1.6), ('step: bus, all-RavKav volumes', PURPLE, '--', 1.4),
                             ('hybrid: bus (calibrated survey)', BLUE, '-', 2.2), ('ticketing: bus (RavKav × OnBoard)', ORANGE, '-', 2.2)]:
        ax.stairs(LF[k][di], x, color=color, linestyle=ls, linewidth=lw, label=k)
    ax.stairs(LF['hybrid: bus (calibrated survey)'][di] + LF['hybrid: taxi-type'][di] + LF['hybrid: rail (survey)'][di], x, color=BLUE, linewidth=1, linestyle='-.', alpha=0.7, label='hybrid transit incl. taxi-type + rail')
    ax.stairs(LF['ticketing: bus (RavKav × OnBoard)'][di] + LF['ticketing: train (station)'][di], x, color=ORANGE, linewidth=1, linestyle='-.', alpha=0.7, label='ticketing transit incl. train')
    ax.set_xlim(0, len(SEQ) - 1); ax.grid(True, color=GRID, linewidth=0.6, axis='y'); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2); ax.set_ylabel('trips crossing the link, 6:00–9:00', color=INK2)
    ax.set_title(f'Direction {d}  ({names[SEQ[0]] if d == "1→23" else names[SEQ[-1]]} → {names[SEQ[-1]] if d == "1→23" else names[SEQ[0]]})', color=INK, fontsize=12)
    ax.legend(frameon=False, fontsize=9, loc='upper right' if d == '1→23' else 'upper left')
axes[1].set_xticks(x); axes[1].set_xticklabels([f'{a} · {names[a]}' for a in SEQ], rotation=55, ha='right', fontsize=9, color=INK2)
fig.suptitle('Corridor bus / transit profiles — calibrated survey (hybrid) vs ticketing, and the calibration steps in between', color=INK, fontsize=14, y=0.995)
plt.tight_layout(); fig.savefig('Output/figures/corridor_profile_hybrid_vs_ticketing.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

## Which area pairs make the difference

For each direction, the contribution of an area pair to the profile is its flow × the number of links it crosses. The table lists the pairs with the largest positive (hybrid > ticketing) and negative (ticketing > hybrid) contributions to the transit profile, and the same aggregated by origin and by destination area.

In [ ]:
pos = {a: i for i, a in enumerate(SEQ)}
Hm = (H_bus + H_taxi + H_rail).reindex(index=SEQ, columns=SEQ).fillna(0); Tm = (T_bus + T_train).reindex(index=SEQ, columns=SEQ).fillna(0)
rows = []
for o in SEQ:
    for d in SEQ:
        if o == d: continue
        nl = abs(pos[d] - pos[o])
        rows.append({'origin': names[o], 'destination': names[d], 'direction': '1→23' if pos[d] > pos[o] else '23→1', 'links crossed': nl,
                     'hybrid trips': Hm.loc[o, d], 'ticketing trips': Tm.loc[o, d], 'diff trips': Hm.loc[o, d] - Tm.loc[o, d], 'diff link-trips': (Hm.loc[o, d] - Tm.loc[o, d]) * nl})
pairs = pd.DataFrame(rows); pairs.to_csv(f'{OUT}/corridor_profile_pair_contributions.csv', index=False, float_format='%.1f')
print("largest contributions to the difference (hybrid − ticketing), link-trips:")
print(pd.concat([pairs.sort_values('diff link-trips').head(10), pairs.sort_values('diff link-trips').tail(10)]).round(0).to_string(index=False))
by_o = pairs.groupby('origin')[['hybrid trips', 'ticketing trips', 'diff link-trips']].sum().sort_values('diff link-trips')
by_d = pairs.groupby('destination')[['hybrid trips', 'ticketing trips', 'diff link-trips']].sum().sort_values('diff link-trips')
print("\nby origin area (corridor-internal transit trips and link-trip difference):"); print(by_o.round(0).to_string())
print("\nby destination area:"); print(by_d.round(0).to_string())

## Transit share along the line — both matrix sets

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5.5), facecolor='white')
for d, di, ls in [('1→23', 0, '-'), ('23→1', 1, '--')]:
    ax.stairs(tbl[f'transit share hybrid {d}'].values, x, color=BLUE, linestyle=ls, linewidth=2, label=f'hybrid set (car + bus + rail), {d}')
    ax.stairs(tbl[f'transit share ticketing {d}'].values, x, color=ORANGE, linestyle=ls, linewidth=2, label=f'ticketing set (car / other + bus + train), {d}')
ax.set_xticks(x); ax.set_xticklabels([f'{a} · {names[a]}' for a in SEQ], rotation=55, ha='right', fontsize=9, color=INK2)
ax.set_xlim(0, len(SEQ) - 1); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.grid(True, color=GRID, linewidth=0.6, axis='y'); ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_color(AXIS)
ax.tick_params(colors=INK2); ax.set_ylabel('transit share of link flow', color=INK2)
ax.set_title('Transit share of the corridor link flow — hybrid set vs ticketing set (the ticketing set\'s total includes walk / other modes)', color=INK, fontsize=12)
ax.legend(frameon=False, fontsize=9, ncol=2)
plt.tight_layout(); fig.savefig('Output/figures/corridor_profile_transit_share.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()
tbl[['link', 'transit share hybrid 1→23', 'transit share ticketing 1→23', 'transit share hybrid 23→1', 'transit share ticketing 23→1']].round(3)

## Why the Nazareth end differs: ticketing coverage is a local-trip problem

The coverage guard of `THS_2017_two_mode_matrix.ipynb` compared **all** bus trips per origin superzone. Splitting them into local (inside the superzone) and inter-superzone trips shows where the ticketing shortfall actually sits.

In [ ]:
keys_raw = pd.read_csv('Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv', encoding='windows-1255')
sz = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
sv_taz = load('Output/ths2017/two_mode/bus_survey_taz.csv'); rk_taz = load('Output/bus/bus_od_taz_new.csv').reindex(index=sv_taz.index, columns=sv_taz.index).fillna(0)
def to_szm(m):
    o = m.index.map(sz); d = m.columns.map(sz); return m.groupby(o).sum().T.groupby(d).sum().T
S, R = to_szm(sv_taz), to_szm(rk_taz)
HAIFA = [12, 13, 14, 15, 16, 17, 18]
fac = pd.read_csv('Output/ths2017/two_mode/bus_calibration_factors_sz.csv').set_index('origin SZ')
rows = []
for z in [19, 4, 23, 30, 34, 33, 37, 17, 14, 16, 9]:
    rows.append({'SZ': z, 'localities': fac.loc[z, 'main localities'], 'guarded': fac.loc[z, 'flag'].startswith('ticketing'),
                 'survey local': S.loc[z, z], 'RavKav local': R.loc[z, z], 'ratio local': R.loc[z, z] / S.loc[z, z],
                 'survey inter-SZ': S.loc[z].sum() - S.loc[z, z], 'RavKav inter-SZ': R.loc[z].sum() - R.loc[z, z], 'ratio inter-SZ': (R.loc[z].sum() - R.loc[z, z]) / (S.loc[z].sum() - S.loc[z, z]),
                 'survey → Haifa SZs': S.loc[z, HAIFA].sum(), 'RavKav → Haifa SZs': R.loc[z, HAIFA].sum()})
cov = pd.DataFrame(rows); cov.to_csv(f'{OUT}/corridor_profile_coverage_local_vs_intercity.csv', index=False, float_format='%.2f')
cov.round(2)

## Findings

1. **Towards Nazareth (1 → 23) the two bus profiles agree.** Once the survey bus is calibrated, its link flows sit within about 10 % of the ticketing bus along the whole Haifa segment (Tirat Carmel → Hadar Carmel). The raw 2018 survey was 25–40 % lower; the calibration closed the gap. The hybrid **transit** profile is nevertheless 1.5 × the ticketing one in this direction, and the reason is the **taxi-type layer** (shared taxis, 9,300 sub-area trips): it adds 1,000–1,400 trips to every Haifa-segment link, and ticketing does not see shared taxis at all.
2. **Towards Tirat Carmel (23 → 1) the ticketing profile is about double** from Ein Hayam to Kiryat Bialik South and three times at the Nazareth end. The pair table shows why: journeys from the Nazareth Area (1,363 ticketed vs 417 survey-based, contributing −9,700 link-trips), from Hamifrats (782 vs 443) and from Neve Yosef (741 vs 250) into Haifa's western districts (Bat Galim, Neve David, Neot Peres, Lower City). Two frames meet here: Hamifrats and Neve Yosef are transfer hubs where ticketing places journeys at the boarding point, and the Nazareth Area is where the coverage guard kept the survey's volumes.
3. **The coverage gap is a local-trip gap.** In the Nazareth superzone ticketing records 8 % of the survey's *local* bus trips but 34 % of its inter-superzone trips and 74 % of its trips to the Haifa superzones; the other guarded superzones show the same shape (local ratios 0.05–0.24, inter-superzone 0.31–1.22). The under-coverage sits in local bus travel — town services, cash fares or unticketed operators — while the intercity market that matters for the corridor is reasonably covered. Applying the guard to whole superzone rows therefore keeps survey volumes on the intercity market too, which is why the hybrid profile is light at the Nazareth end. The refinement is to apply the coverage rule separately to local and inter-superzone trips (added to the task list); with it the 23 → 1 profile would move part-way towards the ticketing one.
4. **Transit share of link flow** is 30–45 % in the hybrid set across the Haifa segment in both directions, against 15–27 % (1 → 23) and 35–55 % (23 → 1) in the ticketing set, whose total also contains walk / other modes. The Krayot–Nazareth segment is where the sets disagree most: the hybrid set has almost no transit heading towards Nazareth in the morning, the ticketing set 60–90 % transit heading towards Haifa — the latter is the combination of hub-attributed journeys and a thin car layer at that end of the earlier matrix.
5. **Rail is immaterial to both profiles** (survey rail: no corridor-internal trips; station-based train: 763 sub-area trips, concentrated at station areas).

## Notes

- **Two frames, not two estimates of one thing.** The ticketing profile counts every rider who boarded in a corridor area, including non-residents and journeys that start with a transfer at a hub (Hamifrats, Neve Yosef); the hybrid profile counts residents from their doorstep. Where they differ most is where those frames differ most.
- **Coverage guard.** In the hybrid set the Nazareth and Shefa-'Amr superzones keep survey bus volumes (ticketing sees only 0.2–0.4 of them), grown to 2022; the ticketing profile carries only the ticketed journeys there.
- **Rail.** Door-to-door survey rail is tiny inside the corridor (no corridor-internal trips at all); the station matrix puts train trips at station areas. Both are small against bus.
- **Transit share** in the ticketing set is measured against a total that includes walking and other modes, so it sits lower by construction; the hybrid set's total is car + bus + rail only.